# PYHF Documents


### Machinery stuff:

Everything you need to know about the package: https://pyhf.readthedocs.io/en/v0.7.6/
, including:
* Installation instructions: https://pyhf.readthedocs.io/en/v0.7.6/installation.html
* Some technical examples: https://pyhf.readthedocs.io/en/v0.7.6/examples.html

### Maths stuff:

* Asymptotic formulae for likelihood-based tests of new physics : 
https://arxiv.org/abs/1007.172

* More slides by the same author: https://www.pp.rhul.ac.uk/~cowan/stat/aachen/cowan_aachen13_4.pdf (page 5)

* The CLs method: https://inspirehep.net/literature/532312 (If you're at Fermilab, you can ask Tom Junk about it directly ;)  )

# Very Short Summary of Upper Limits Setting Procedure 

For MC study, the upper limits are set under the assumption of no detected signals, so that the results can be directly compared against existing exlusion limits derived from data. 

The limits setting procedure employs the likelihood-based hypothesis test. The null and test hypotheses are formally written as:
\begin{align}
    Null\ hypothesis:\ H_{s+b} = H (\mu = 1, \theta),\\
    Test\ hypothesis:\ H_{b} = H (\mu = 0, \theta).
\end{align}
The parameter $\mu$ determines the strength of the signal process, where $\mu = 0$ corresponds to the background-only $H_{b}$ hypothesis and $\mu = 1$ corresponds to the nominal signal $H_{s+b}$ hypothesis.
Other parameters in the hypothesis represent nuisance parameters, denoted as $\theta$.

To exclude the null hypothesis $H_{s+b}$ at some Confidence Levels (CL), a test statistic is performed for a hypothesised $\mu$ against $H_{s+b}$ and $H_{b}$.
Likelihood-based functions are chosen to construct the test statistic for a multi-binned histogram.
The likelihood function is the product of Poisson probabilities of all bins in the histogram:

\begin{equation}
L(\mu, \theta) =  \prod_{i=1}^{N} \frac{(\mu s_i + b_i)^{n_i}}{n_i!} e^{-(\mu s_i + b_i)}  \prod_{\theta\in\Theta} c_\theta(a_\theta|\theta).
\end{equation}

The test statitics of interest is the **upper limits (qtilde)** defined in Section 3.6 in _Asymptotic formulae for likelihood-based tests of new physics_.
Highly reccomend reading so that you can follow each step.

# Import Stuff

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rc('font', **{'size':14})

import warnings
warnings.filterwarnings("ignore")

import pyhf
from pyhf.contrib.viz import brazil

pyhf.set_backend("numpy")

<h1> Inputs </h1>

Here are some MC info as an example, which are the arrival time distributions of signals and backgrounds. 
Likelihood functions for hypothesis testing are constructed from these distributions.
This dataset here is from an end-to-end MC study with reconstruction/selection already applied. 

#### Signal: Heavy Neutral Lepton with mass of 240 MeV

* Signal strength squared: The coupling $|U_{\mu4}|^2$ of HNL
* Signal mass: HNL mass in MeV
* Signal: Arrival time distribution in ns
* Statistical error
* Flux error: Booster Neutrino Beam flux error, mainly driven by kaon parents

**Note:** The signal to background ratio or the signal strength is very imporatnt to the performance of the limits setting. More on it later in the tutorial.

#### Background: Neutrinos and Cosmics

* Background: Arrival time distribution in ns
* Statistical error 
* Flux error: Booster Neutrino Beam flux error
* Cross-section error: Neutrino cross-section error, provided by GENIEReweight

This dataset will be shown for different studies:
1. For a truth-only study containing only statistics error
2. For a more complete MC study containing more errors like flux/xsec/etc.

In [ ]:
bkg = [1.29, 4.13, 8.25, 6.72, 102.93, 314.91, 754.57, 1153.73, 1704.55, 1585.77, 1125.26, 729.31, 373.32, 128.0, 103.96, 12.14, 21.68, 16.51, 5.42]

bkg_stat_err = [1.29, 4.13, 5.84, 4.51, 20.34, 49.73, 84.12, 92.79, 120.18, 112.66, 80.14, 77.31, 51.18, 29.36, 34.47, 6.25, 8.65, 8.25, 4.32]

bkg_flx_err = [0.11, 0.3, 0.88, 0.63, 9.0, 31.39, 45.68, 66.49, 87.93, 88.57, 67.67, 34.54, 19.13, 6.06, 3.84, 0.76, 1.4, 1.04, 0.54]

bkg_xsec_err = [1.11, 3.31, 4.14, 2.34, 34.42, 80.32, 167.25, 270.13, 404.86, 379.47, 289.99, 157.14, 67.21, 21.21, 15.27, 6.71, 6.9, 13.25, 4.05]

In [ ]:
signal_strength_squared = 4.64e-08

signal_m = 240 #MeV

signal = [16.24, 14.71, 14.07, 13.33, 12.17, 11.03, 10.79, 11.82, 14.91, 19.82, 25.8, 33.47, 38.44, 38.36, 37.08, 28.41, 24.81, 21.6, 19.18]

signal_stat_err = [0.63, 0.6, 0.59, 0.57, 0.55, 0.52, 0.52, 0.54, 0.61, 0.7, 0.8, 0.91, 0.97, 0.97, 0.96, 0.84, 0.78, 0.73, 0.69]

signal_flx_err = [1.17, 1.06, 0.98, 0.94, 0.85, 0.77, 0.77, 0.83, 1.01, 1.35, 1.72, 2.24, 2.57, 2.57, 2.5, 1.95, 1.73, 1.52, 1.35]

<h1>Plot Inputs</h1>

In [ ]:
fig, (ax1, ax2) = plt.subplots(1,2, figsize = (10,4))

xlabel = "Arrival Time [ns]"
ylabel = "Entries / 1 ns"
#-----------------------------------------------------#
#Define x-axis for the arrival time distribution
bins = np.arange(0,20,1)
bins_mid = np.convolve(bins, [0.5, 0.5], "valid")
#-----------------------------------------------------#
#Plot ax1
ax1.step(bins, np.insert(bkg, 0, 0), color = 'tab:blue', label =  "Background")
ax1.step(bins, np.insert(signal, 0, 0), color = 'tab:red', label =  "Signal")

ax1.set_xlim(0, 19)
ax1.set_ylim(0, 3500)
ax1.set_xlabel(xlabel)
ax1.set_ylabel(ylabel)
#-----------------------------------------------------#
#Plot ax2
ax2.step(bins, np.insert(bkg, 0, 0), color = 'tab:blue', label =  "Background")
ax2.step(bins, np.insert(signal, 0, 0), color = 'tab:red', label =  "Signal")

ax2.set_xlim(0, 19)
ax2.set_ylim(0, 70)

ax2.set_xlabel(xlabel)
ax2.set_ylabel(ylabel)
#-----------------------------------------------------#
#Uncomment to plot errorbars
#ax1.errorbar(bins_mid, bkg, yerr=bkg_xsec_err,  capsize=5, fmt = 'none', label = 'Cross Section', color = 'tab:cyan')
#ax1.errorbar(bins_mid, bkg, yerr=bkg_flx_err,  capsize=5, fmt = 'none', label = 'Flux', color = 'tab:green')
#ax1.errorbar(bins_mid, bkg, yerr=bkg_stat_err,  capsize=5, fmt = 'none', label = 'Statistical', color = 'tab:pink')

#ax1.errorbar(bins_mid, signal, yerr=signal_flx_err,  capsize=5, fmt = 'none', color = 'tab:green')
#ax1.errorbar(bins_mid, signal, yerr=signal_stat_err,  capsize=5, fmt = 'none', color = 'tab:pink')

#ax2.errorbar(bins_mid, bkg, yerr=bkg_xsec_err,  capsize=5, fmt = 'none', label = 'Cross Section', color = 'tab:cyan')
#ax2.errorbar(bins_mid, bkg, yerr=bkg_flx_err,  capsize=5, fmt = 'none', label = 'Flux', color = 'tab:green')
#ax2.errorbar(bins_mid, bkg, yerr=bkg_stat_err,  capsize=5, fmt = 'none', label = 'Statistical', color = 'tab:pink')

#ax2.errorbar(bins_mid, signal, yerr=signal_flx_err,  capsize=5, fmt = 'none', color = 'tab:green')
#ax2.errorbar(bins_mid, signal, yerr=signal_stat_err,  capsize=5, fmt = 'none', color = 'tab:pink')
#-----------------------------------------------------#
ax1.legend(loc = 'upper left',fontsize = 12, ncol = 2)
ax2.legend(loc = 'upper left',fontsize = 12, ncol = 2)
#-----------------------------------------------------#
fig.tight_layout()
plt.show()

<h1>Define Models</h1>

The `model` is configured for one set of `samples`, made up of **signal** and **background**.

For each respective signal and background sample, there are `modifiers` to configure how the input can be varied.

A summary of different types of modifier is provided here: https://pyhf.readthedocs.io/en/v0.7.6/likelihood.html#modifiers

#### Important modifer for the signal strength:

Specifically for **signal**, there must be a modifer 
`{"name": "mu", "type": "normfactor", "data": None},`.
This is the **scaling factor $\mu$** on the signal strength to set upper limits on.

In this HNL example,  $\mu$ = 1 is equivalent to the input coupling $|U_{\mu4}|^2$ = $4.64\times 10^{-8}$. 

#### Important configuration for MC study: 

`data = bkg + model.config.auxdata`

This line is specifically for MC study, where `data` is assumed to be the same as `bkg` i.e. the input predicted background.

#### Two example models are shown below:
1. For a truth-only study containing only statistics error
2. For a more complete MC study containing more errors like flux/xsec/etc.

### 1. Truth-only study modifiers:

For a truth-only study, we assume there are only statistical errors on the inputs.
There are types of modifiers that can be used for statistical errors:
1. `shapesys`: Poisson shape, no sample correlation, no bin correlation

Can be used for sample with a relatively low rate, and therefore follows a Poisson distirbution. 
In this case, we apply it to signal statiscal error.

2. `staterror`: Gaussian shape, no sample correlation, no bin correlation

This treatment of statistical uncertainty employs a modified version of the Beeston-Barlow method to account for statistical fluctuations due to finite statistics. 
The modifier follows a Gaussian shape for bins with high statistics and falls back to a Poisson shape for bins with low statistics.
In this case, we apply to background statistical error since we expect the distribution to vary between high to low statistics.

Some reading:
* Beeston-Barlow mathematical model: https://www.sciencedirect.com/science/article/pii/001046559390005W
* HistFactory: https://cds.cern.ch/record/1456844?ln=en

In [ ]:
def make_model_truth(signal, bkg):
    model = pyhf.Model(
        {
      "channels": [
        {
          "name": "singlechannel",
          "samples": [
            {
              "name": "signal",
              "data": signal,
              "modifiers": [
                {"name": "mu", "type": "normfactor", "data": None},
                {"name": "signal_stat", "type": "shapesys", "data": signal_stat_err},
              ]
            },
            {
              "name": "background",
              "data": bkg,
              "modifiers": [
                {"name": "bkg_stat", "type": "staterror", "data": bkg_stat_err},
              ]
            }
          ]
        }
      ]
    }
    )

    print(f'Samples:\n {model.config.samples}')
    print(f'Modifiers are:\n {model.config.modifiers}')

    #For MC study: we assume data is the same as predicted background
    data = bkg + model.config.auxdata
    
    return model, data

### 2. Complete MC study modifiers:

In additional to statistical errors, there are other errors like flux and cross-section errors.
Both these errors can be bin-correlated so another modifier is needed.
From the list of provided modifiers, the suitable one is:

* `histosys`: Gaussian shape, no sample correlation, has bin correlation

The modifier needs 2 histograms inputs, low = central value - error and high = central value + error.

In [ ]:
def make_model_full_systematics(signal, bkg):
    
    signal_flx_lo = (np.array(signal) - np.array(signal_flx_err)).tolist()
    signal_flx_hi = (np.array(signal) + np.array(signal_flx_err)).tolist()
    
    bkg_flx_lo = (np.array(bkg) - np.array(bkg_flx_err)).tolist()
    bkg_flx_hi = (np.array(bkg) + np.array(bkg_flx_err)).tolist()
    
    bkg_xsec_lo = (np.array(bkg) - np.array(bkg_xsec_err)).tolist()
    bkg_xsec_hi = (np.array(bkg) + np.array(bkg_xsec_err)).tolist()
    
    
    model = pyhf.Model(
        {
      "channels": [
        {
          "name": "singlechannel",
          "samples": [
            {
              "name": "signal",
              "data": signal,
              "modifiers": [
                {"name": "mu", "type": "normfactor", "data": None},
                {"name": "signal_stat", "type": "shapesys", "data": signal_stat_err},
                {"name": "signal_flx", "type": "histosys", "data": {"lo_data": signal_flx_lo, "hi_data": signal_flx_hi} },
              ]
            },
            {
              "name": "background",
              "data": bkg,
              "modifiers": [
                {"name": "bkg_stat", "type": "staterror", "data": bkg_stat_err},
                {"name": "bkg_flx", "type": "histosys", "data": {"lo_data": bkg_flx_lo, "hi_data": bkg_flx_hi} },
                {"name": "bkg_xsec", "type": "histosys", "data": {"lo_data": bkg_xsec_lo, "hi_data": bkg_xsec_hi} },
              ]
            }
          ]
        }
      ]
    }
    )

    print(f'Samples:\n {model.config.samples}')
    print(f'Modifiers are:\n {model.config.modifiers}')

    data = bkg + model.config.auxdata
    
    return model, data

In [ ]:
model, data = make_model_truth(signal, bkg)

#model, data = make_model_full_systematics(signal, bkg)

<h1>Set Upper Limits</h1>

The function `pyhf.infer.intervals.upper_limits.upper_limit` is the key function to set an **upper limits** on the signal strength.


https://pyhf.readthedocs.io/en/v0.7.6/_generated/pyhf.infer.intervals.upper_limits.upper_limit.html#pyhf.infer.intervals.upper_limits.upper_limit

#### Important configuration:

* `poi_vals`: The range to sample the **scaling factor $\mu$**. For the provided example dataset, the acceptable range is 0 to 0.4.
* `level`: Confidence level (CL), where 0.1 = 1 - 0.9, equivalent to a CL of 90%
* `test_stat = 'qtilde'`: **DO NOT CHANGE.**

#### Fitting explanation:

The function samples through every value in `poi_vals` to calculate the limits and the corresponding CL.
Once the sampling is done, it performs an interpolation at the CL value of choice (0.1 in this case) to determine the limits.
So the more fined the spacing of `poi_vals`, the better the interpolation is.


#### Output explanation:
* `obs_limit_single` : Observed limit based on the input data. Since we set data to be the same as the predicted background, the observed limit here is the same as `exp_limits_single` at the specified CL.
* `exp_limits_single` : Expected limits based on the input predicted background, 5 values at [-2, -1, 0, 1, 2] sigmas from the specified CL.
* `scan` : The sample range, the same as `poi_vals`.
* `results` : Expected limits across all the sample range.

In [ ]:
poi_vals = np.linspace(0, 0.4, 20)
print(poi_vals)

In [ ]:
obs_limit_single, exp_limits_single, (scan, results) = pyhf.infer.intervals.upper_limits.upper_limit(
                                                                            data, 
                                                                            model, 
                                                                            poi_vals, 
                                                                            level=0.1, 
                                                                            return_results=True,
                                                                            test_stat='qtilde')

# Brazil Plot

x-axis = `poi_vals`

y-axis = `results` at various CL


What to look for:

- Yellow/green bands corresponding to CL at [-2, -1, 0, 1, 2] sigmas. The bands should be smooth and clearly identified, not rugged or squished. If it is, try adjusting the poi_vals.
- 1 red line corresponding to the CL of choice

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(7, 5)
brazil.plot_results(poi_vals, results, ax=ax, test_size=0.10,)
fig.show()

# Relationship of $\mu$ and Signal Strength

The output `exp_limits_single` is the **scaling factor $\mu$** and NOT the signal strength.
Need to convert from $\mu$ to signal strength, such that

\begin{equation}
Scaled\ Signal\ Strength = \mu \times Input\ Signal\ Strength
\end{equation}

**Note:** The HNL example is a special case because its signal strength is the square of the coupling $|U_{\mu4}|^2$. Therefore, need to take a square root of the scaling factor $\mu$.


#### Importance on the input signal strength! 

The range of `poi_vals` depends heavily on your signal to background ratio i.e. the signal strength.
Since hypotheses are defined for 0 < $\mu$ < 1, the range of `poi_vals` should stay within [0, 1] so that the fitting can perform accurately.

Since $\mu$ is essentially _a scaled signal strength_, it is very important that the input signal strength should result in  0 < $\mu$ < 1, equivalent to having the input signal strength very close to the expected limits.

If you do not know what to input for signal strength, it is reccomnded to do some trials and errors:

- Input some guessed signal strength
- Perform the fitting to acquire the expected limits
- Repeat the process with the expected limits as the new signal strength
- Check the Brazil plots until you clearly see the green/yellow bands and the red line corresponding to the CL.

In [ ]:
exp_limits_signal = []

for l in exp_limits_single:
    
    s = np.sqrt(l) * signal_strength_squared
     
    #s = l * signal_strength
    
    exp_limits_signal.append(s)

<h1>Plot Upper Limits</h1>

In [ ]:
fig, ax1 = plt.subplots(1,1, figsize=(8,6))

plt.grid(axis = 'both', color='gainsboro', linestyle = ":")
#-------------------------------------------------------------------
ax1.scatter(signal_m, exp_limits_signal[0], c = 'gold', marker='x', s = 100, label = r'CL$_{s} \pm 2 \sigma$')
ax1.scatter(signal_m, exp_limits_signal[1], c = 'green', marker='x', s = 100, label = r'CL$_{s} \pm 1 \sigma$')
ax1.scatter(signal_m, exp_limits_signal[2], c = 'black', marker='x', s = 100, label=r'CL$_{s, expected}$')
ax1.scatter(signal_m, exp_limits_signal[3], c = 'green', marker='x', s = 100)
ax1.scatter(signal_m, exp_limits_signal[4], c = 'gold', marker='x', s = 100)

ax1.set_yscale('log')
ax1.legend()

ax1.set_ylabel(r"$|U_{\mu4}|^2$")
ax1.set_xlabel("Mass [MeV]")
#-------------------------------------------------------------------
fig.tight_layout()

plt.show()